# Mangrove Canopy Height, AGB, and Carbon Stock: Data Preprocessing

**Author** : Muhammad Wahyu Ramadhan  
**GitHub**  : github.com/mwahyur46  
**LinkedIn**: linkedin.com/in/mwahyur

This notebook prepares the predictor stack and training data for the
two-stage mangrove canopy height (CH) and above-ground biomass (AGB)
regression pipeline, using the Google Earth Engine Python API.

**Workflow steps:**
1. Authenticate and initialize Earth Engine
2. Build the Sentinel-2 cloud-masked median composite
3. Compute spectral indices (NDVI, NDWI, MNDWI, NDMI, CMRI, MVI, NDRE, SAVI, EVI)
4. Build the Sentinel-1 speckle-filtered VV/VH composite
5. Compute SRTM slope
6. Build the Global Mangrove Watch (GMW) mask
7. Assemble the full predictor stack (S2 + indices + S1 + slope)
8. Sample GEDI L2A (canopy height) and GEDI L4A (AGB) targets
9. Export training samples (CSV) and the predictor stack (GeoTIFF) to Drive

**Feature set (full version):** S2 bands, 9 spectral indices, S1 VV/VH, SRTM slope.
This is the full predictor set; a reduced public version (S2 + indices only)
will be derived later once results are validated.

**References:** see `src/gee_utils.py` module docstring for full citations.


In [1]:
# pip install earthengine-api geemap pandas (uncomment if needed)
# %pip install earthengine-api geemap pandas


In [2]:
import sys
sys.path.append('../src')

import ee
import gee_utils as geu
import geopandas as gpd
import json
import shapely

ee.Authenticate()
ee.Initialize(project='mwahyur')


## 1. Configuration and AOI

Area of interest: West Kalimantan Mangrove Coast (south of Pontianak).
Replace the placeholder asset path below with your own AOI asset, or define
the geometry directly with `ee.Geometry.Polygon(...)`.


In [ ]:
# ============================================================
# CONFIG
# ============================================================
S2_START = '2025-01-01'
S2_END = '2025-12-31'
GEDI_START = '2019-01-01'
GEDI_END = '2025-06-01'
CLOUD_PCT = 20
SMOOTHING_RADIUS = 20
SCALE = 25
NUM_PIXELS = int(1e6)
SEED = 42
TILE_SCALE = 8
EXPORT_FOLDER = 'GEE_exports'
# Stack export data name prefix
EXPORT_NAME = 'kalimantan_stack_2025'

# AOI: choose one of the following options to load the AOI geometry
# Option A: GEE asset
# TODO: replace with your own AOI asset path
AOI_ASSET = 'projects/mwahyur/assets/aoi_west_kalimantan_mangrove'
aoi = ee.FeatureCollection(AOI_ASSET).geometry()

# Option B: from local GeoJSON file
# AOI_PATH = '../data/aoi/aoi.geojson'
# gdf = gpd.read_file(AOI_PATH).to_crs('EPSG:4326')
# aoi = ee.Geometry(json.loads(shapely.to_geojson(gdf.geometry.union_all())))

print('AOI loaded.')
print('AOI area (km2):', aoi.area().divide(1e6).getInfo())


AOI loaded.
AOI area (km2): 6963.558685596158


## 2. Sentinel-2 Median Composite

Cloud-masked using the Scene Classification Layer (SCL), then reduced to a
median composite over the AOI.


In [4]:
s2_median = geu.get_s2_median(aoi, S2_START, S2_END, cloud_pct=CLOUD_PCT)

print('S2 median composite bands:', s2_median.bandNames().getInfo())


S2 median composite bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']


## 3. Spectral Indices

Nine indices computed from the S2 median composite: NDVI, NDWI, MNDWI, NDMI,
CMRI (Gupta et al. 2018), MVI (Baloloy et al. 2020), NDRE, SAVI, EVI.


In [5]:
indices = geu.compute_indices(s2_median)

print('Index bands:', indices.bandNames().getInfo())


Index bands: ['NDVI', 'NDWI', 'MNDWI', 'NDMI', 'CMRI', 'MVI', 'NDRE', 'SAVI', 'EVI']


## 4. Sentinel-1 SAR Composite

Speckle-filtered VV/VH composite (IW mode, descending orbit, 10 m resolution).


In [6]:
s1 = geu.get_s1_filtered(aoi, S2_START, S2_END, smoothing_radius=SMOOTHING_RADIUS)

print('S1 bands:', s1.bandNames().getInfo())


S1 bands: ['VV', 'VH']


## 5. Topography (SRTM Slope)


In [7]:
slope = geu.get_slope(aoi)

print('Topography band:', slope.bandNames().getInfo())


Topography band: ['slope']


## 6. Mangrove Mask (Global Mangrove Watch v3, 2020)


In [8]:
gmw_mask = geu.get_gmw_mask(aoi)

print('GMW mask built.')


GMW mask built.


## 7. Base Predictor Stack

S2 + indices + S1 + slope, assembled wall-to-wall (not masked to mangrove
extent here, to avoid dropping GEDI footprints during sampling due to
slight raster/footprint misalignment at the GMW mask boundary).


In [9]:
base_stack, base_bands = geu.build_base_stack(
    aoi, S2_START, S2_END, cloud_pct=CLOUD_PCT, smoothing_radius=SMOOTHING_RADIUS
)

print('Base feature stack bands:', base_bands)
print('Total predictors:', len(base_bands))


Base feature stack bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'MNDWI', 'NDMI', 'CMRI', 'MVI', 'NDRE', 'SAVI', 'EVI', 'VV', 'VH', 'slope']
Total predictors: 22


## 8. Stage 1 Target: Canopy Height (GEDI L2A RH98)

RH98: height at which 98% of returned energy is below. Filtered to
`quality_flag == 1` (good shots only), masked to mangrove extent.


In [ ]:
rh98_target, gedi_l2a = geu.get_ch_target(aoi, GEDI_START, GEDI_END, gmw_mask)

print('GEDI L2A collection size:', gedi_l2a.size().getInfo())
print('RH98 valid pixel count:', rh98_target.reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=SCALE, maxPixels=1e9
).getInfo())


GEDI L2A collection size: 122
RH98 valid pixel count: {'rh98': 61282}


### 8a. Sample CH Training Data


In [11]:
ch_training_image = base_stack.updateMask(gmw_mask).addBands(rh98_target.rename('rh98'))

ch_samples = geu.sample_training_data(
    ch_training_image, aoi, scale=SCALE, num_pixels=NUM_PIXELS,
    seed=SEED, tile_scale=TILE_SCALE,
)

# print('CH total samples:', ch_samples.size().getInfo())  # skip, trigger eager eval


## 9. Stage 2 Target: AGB (GEDI L4A AGBD)

Quality filter: mean `l4_quality_flag > 0.3` AND `agbd > 0`, masked to
mangrove extent. Note: in the GEE JS version, the AGB feature stack also
includes the Stage 1 CH prediction (`CH_m`) as a predictor. Since the CH
model is trained locally in this Python pipeline (notebook `02`), the CH
band is added to the AGB feature stack downstream, after Stage 1 inference
in notebook `04`. Here we export the base stack and AGB target/samples
without `CH_m` -- `CH_m` is appended as a feature during Stage 2 training
using the Stage 1 model's predictions.


In [12]:
agbd_target, gedi_l4a_col = geu.get_agb_target(
    aoi, GEDI_START, GEDI_END, gmw_mask, quality_threshold=0.3
)

# print('GEDI L4A collection size:', gedi_l4a_col.size().getInfo())
# print('AGBD valid pixel count:', agbd_target.reduceRegion(
#     reducer=ee.Reducer.count(), geometry=aoi, scale=SCALE, maxPixels=1e9
# ).getInfo())


### 9a. Sample AGB Training Data

Sampled from the base stack (without `CH_m`). `CH_m` will be joined to this
table locally in Python (notebook `03`) using Stage 1 model predictions at
the same sample locations, since GEE-side sequential dependency between
stages is avoided in this hybrid pipeline.


In [13]:
agb_training_image = base_stack.updateMask(gmw_mask).addBands(agbd_target.rename('agbd'))

agb_samples = geu.sample_training_data(
    agb_training_image, aoi, scale=SCALE, num_pixels=NUM_PIXELS,
    seed=SEED, tile_scale=TILE_SCALE,
)

# print('AGB total samples:', agb_samples.size().getInfo()) # skip, trigger eager eval


## 10. Export

Two sample tables (CH, AGB) exported as CSV, plus the wall-to-wall base
predictor stack exported as GeoTIFF for final inference (notebook `04`).

Exports run asynchronously in Earth Engine. Monitor progress at
https://code.earthengine.google.com/tasks or via `task.status()`.


In [ ]:
ch_task = geu.export_samples_to_drive(
    ch_samples, description='ch_samples_full', folder=EXPORT_FOLDER
)

agb_task = geu.export_samples_to_drive(
    agb_samples, description='agb_samples_full', folder=EXPORT_FOLDER
)

stack_task = geu.export_image_to_drive(
    base_stack.updateMask(gmw_mask), description=EXPORT_NAME, aoi=aoi,
    folder=EXPORT_FOLDER, scale=SCALE,
)

print('Export tasks started:')
print('  CH samples  :', ch_task.id)
print('  AGB samples :', agb_task.id)
print('  Base stack  :', stack_task.id)


Export tasks started:
  CH samples  : VPVBDMT255DGCLPVFNJVI3SS
  AGB samples : 6MWFGRVLV7H2NK4E7YPP77DO
  Base stack  : VSWW6RWVHVOVYW6FNDZOS67U


In [15]:
# Check export status (re-run this cell to poll progress)
for task, name in [(ch_task, 'CH samples'), (agb_task, 'AGB samples'), (stack_task, 'Base stack')]:
    status = task.status()
    print(f"{name:<14}: {status['state']}")


CH samples    : READY
AGB samples   : READY
Base stack    : READY


## Next Steps

Once exports complete, download the files from Google Drive into
`data/raw/`:
- `ch_samples_full.csv`
- `agb_samples_full.csv`
- `base_stack_west_kalimantan.tif`

Then proceed to `02_ch_model_training.ipynb`.
